In [1]:
!pip install transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

dataset = load_dataset("zeroshot/twitter-financial-news-sentiment")

print(dataset)
print(f"Sample Text: {dataset['train'][0]['text']}")
print(f"Label: {dataset['train'][0]['label']}")

README.md:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

sent_train.csv:   0%|          | 0.00/859k [00:00<?, ?B/s]

sent_valid.csv:   0%|          | 0.00/217k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2388
    })
})
Sample Text: $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT
Label: 0


In [3]:
from transformers import AutoTokenizer

model_ckpt = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [4]:
def tokenize_function(examples):

    return tokenizer(examples["text"], truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

print("Tokenized columns:", tokenized_datasets["train"].column_names)

Map:   0%|          | 0/9543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

Tokenized columns: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


In [5]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [6]:
from transformers import AutoModelForSequenceClassification
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Explicitly map the dataset's numerical labels to readable financial sentiment classes
id2label = {0: "bearish", 1: "bullish", 2: "neutral"}
label2id = {"bearish": 0, "bullish": 1, "neutral": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(device)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [12]:
import evaluate
import numpy as np

f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=predictions, references=labels)

    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")

    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

In [13]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./finbert_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.429703,0.388578,0.853434,0.807516
2,0.158447,0.396974,0.877722,0.839033
3,0.137037,0.459594,0.877722,0.838569


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1791, training_loss=0.2966412470734765, metrics={'train_runtime': 368.9011, 'train_samples_per_second': 77.606, 'train_steps_per_second': 4.855, 'total_flos': 760885462064826.0, 'train_loss': 0.2966412470734765, 'epoch': 3.0})

In [14]:
import pandas as pd

eval_metrics = trainer.evaluate()
print("Holdout Metrics:", eval_metrics)

history = pd.DataFrame(trainer.state.log_history)
train_loss = history.dropna(subset=["loss"])[["step", "loss"]]
eval_loss = history.dropna(subset=["eval_loss"])[["step", "eval_loss", "eval_accuracy", "eval_f1"]]

loss_trajectory = pd.merge(train_loss, eval_loss, on="step", how="outer").ffill()
print("\nLoss & Metric History:")
print(eval_loss)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.137037,0.388578,3,0.853434,0.807516


Holdout Metrics: {'eval_loss': 0.38857805728912354, 'eval_accuracy': 0.8534338358458962, 'eval_f1': 0.8075162468974634}

Loss & Metric History:
     step  eval_loss  eval_accuracy   eval_f1
59    597   0.388578       0.853434  0.807516
120  1194   0.396974       0.877722  0.839033
181  1791   0.459594       0.877722  0.838569
183  1791   0.388578       0.853434  0.807516


In [15]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

predictions = trainer.predict(tokenized_datasets["validation"])
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

target_names = ["bearish", "bullish", "neutral"]
print(classification_report(labels, preds, target_names=target_names))
print("\nConfusion Matrix:\n", confusion_matrix(labels, preds))

              precision    recall  f1-score   support

     bearish       0.63      0.87      0.73       347
     bullish       0.83      0.74      0.79       475
     neutral       0.93      0.88      0.91      1566

    accuracy                           0.85      2388
   macro avg       0.80      0.83      0.81      2388
weighted avg       0.87      0.85      0.86      2388


Confusion Matrix:
 [[ 301   14   32]
 [  51  353   71]
 [ 125   57 1384]]


In [16]:
output_model_dir = "./finbert-financial-sentiment"

trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

print(f"Artifacts successfully saved to {output_model_dir}")

from transformers import pipeline
import torch

classifier = pipeline(
    "text-classification",
    model=output_model_dir,
    tokenizer=output_model_dir,
    device=0 if torch.cuda.is_available() else -1
)

sample_phrase = "Operating margins expanded by 120 bps due to aggressive supply chain restructuring."

print("\nInference Test:")
print(f"Text: {sample_phrase}")
print(f"Prediction: {classifier(sample_phrase)}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Artifacts successfully saved to ./finbert-financial-sentiment


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Inference Test:
Text: Operating margins expanded by 120 bps due to aggressive supply chain restructuring.
Prediction: [{'label': 'bearish', 'score': 0.9220540523529053}]
